In [0]:
# =============================================================================
# F1-Pulse | Gold Layer — Driver Performance Analytics
# Notebook: 04_Gold_Analytics
# Author:   Jafar891
# Updated:  2026
#
# Reads Silver enriched_laps, produces aggregated driver performance metrics,
# a constructor standings table, and a lap progression table.
# All Gold tables are dashboard-ready and saved as Delta.
# Idempotent — safe to re-run.
# =============================================================================

import sys
import importlib
import logging

sys.path.append("/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse")

# Reload modules so Databricks picks up edits without a cluster restart
import modules.gold_helpers    as gold_helpers
import modules.gold_transforms as gold_transforms
importlib.reload(gold_helpers)
importlib.reload(gold_transforms)

from modules.gold_helpers import read_silver, write_gold, log_validity_summary
from modules.gold_transforms import (
    build_driver_leaderboard,
    build_constructor_standings,
    build_lap_progression,
)

from config.config import YEAR, CATALOG, SILVER, GOLD

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.gold")


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------

def run_gold() -> None:
    log.info("=" * 60)
    log.info(f"F1-Pulse Gold Analytics — {YEAR}")
    log.info("=" * 60)

    results = {"success": [], "failed": []}

    # ------------------------------------------------------------------
    # 1. Read Silver — reused across all three Gold tables
    # ------------------------------------------------------------------
    log.info("\n[1/4] Reading Silver layer …")
    silver_laps = read_silver(spark, CATALOG, SILVER, "enriched_laps")
    log_validity_summary(silver_laps)

    # ------------------------------------------------------------------
    # 2. Driver Leaderboard
    # ------------------------------------------------------------------
    log.info("\n[2/4] Driver performance leaderboard …")
    try:
        leaderboard = build_driver_leaderboard(silver_laps, season_year=YEAR)
        write_gold(leaderboard, CATALOG, GOLD, "driver_performance_metrics")
        results["success"].append("driver_performance_metrics")
    except Exception as e:
        log.error(f"  ❌ Driver leaderboard failed: {e}")
        results["failed"].append("driver_performance_metrics")

    # ------------------------------------------------------------------
    # 3. Constructor Standings
    # ------------------------------------------------------------------
    log.info("\n[3/4] Constructor standings …")
    try:
        constructors = build_constructor_standings(silver_laps, season_year=YEAR)
        write_gold(constructors, CATALOG, GOLD, "constructor_standings")
        results["success"].append("constructor_standings")
    except Exception as e:
        log.error(f"  ❌ Constructor standings failed: {e}")
        results["failed"].append("constructor_standings")

    # ------------------------------------------------------------------
    # 4. Lap Progression
    # ------------------------------------------------------------------
    log.info("\n[4/4] Lap-by-lap progression …")
    try:
        progression = build_lap_progression(silver_laps, season_year=YEAR)
        write_gold(progression, CATALOG, GOLD, "lap_progression")
        results["success"].append("lap_progression")
    except Exception as e:
        log.error(f"  ❌ Lap progression failed: {e}")
        results["failed"].append("lap_progression")

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    log.info("\n" + "=" * 60)
    log.info("GOLD SUMMARY")
    log.info("=" * 60)
    for t in results["success"]:
        log.info(f"  ✅ {CATALOG}.{GOLD}.{t}")
    for t in results["failed"]:
        log.warning(f"  ❌ {t} — check logs above")
    log.info("=" * 60)

    # Preview Gold tables in notebook output
    # display() and spark are Databricks globals — noqa suppresses linter warnings
    if "driver_performance_metrics" in results["success"]:
        log.info("\n🏎️  Driver Leaderboard")
        display(spark.table(f"{CATALOG}.{GOLD}.driver_performance_metrics"))  # noqa: F821

    if "constructor_standings" in results["success"]:
        log.info("\n🏭  Constructor Standings")
        display(spark.table(f"{CATALOG}.{GOLD}.constructor_standings"))       # noqa: F821

    if "lap_progression" in results["success"]:
        log.info("\n📈  Lap Progression (first 20 rows)")
        display(                                                               # noqa: F821
            spark.table(f"{CATALOG}.{GOLD}.lap_progression").limit(20)
        )

    if results["failed"]:
        raise RuntimeError(
            f"Gold layer completed with errors: {results['failed']}"
        )


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

run_gold()